# NB-603: Vector Databases

Work with vector databases for RAG.

## Learning Objectives
- Qdrant setup
- ChromaDB operations
- Milvus basics
- Vector similarity search

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Libraries imported")

## 1. Simple Vector Store

In [ ]:
class SimpleVectorDB:
    def __init__(self):
        self.documents = []
        self.embeddings = None
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def add(self, documents: list[str]):
        """Add documents to the database."""

        # TODO: Generate embeddings
        new_embeddings = self.embedder.encode(documents)

        # Store
        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k=5) -> list[tuple]:
        """Search for similar documents."""

        if self.embeddings is None:
            return []

        # TODO: Compute similarity
        query_emb = self.embedder.encode([query])[0]
        similarities = np.dot(self.embeddings, query_emb)

        # Get top-k
        top_indices = similarities.argsort()[-top_k:][::-1]

        return [(self.documents[i], similarities[i]) for i in top_indices]

# Test
db = SimpleVectorDB()
db.add(["Machine learning is awesome", "I love Python programming"])

results = db.search("AI and coding")
for doc, score in results:
    print(f"[{score:.3f}] {doc}")

## 2. ChromaDB Integration

In [ ]:
# TODO: ChromaDB setup
try:
    import chromadb

    # Initialize client
    client = chromadb.Client()

    # Create collection
    collection = client.create_collection("documents")

    # Add documents
    documents = ["Doc 1", "Doc 2", "Doc 3"]
    embeddings = SentenceTransformer('all-MiniLM-L6-v2').encode(documents).tolist()

    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=["doc1", "doc2", "doc3"]
    )

    # Query
    results = collection.query(
        query_embeddings=[embeddings[0]],
        n_results=2
    )

    print("ChromaDB test passed")
except ImportError:
    print("ChromaDB not installed")

## 3. Qdrant Integration

In [ ]:
# TODO: Qdrant setup
try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct

    # Initialize client (in-memory)
    client = QdrantClient(location=":memory:")

    # Create collection
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

    # Insert vectors
    client.upsert(
        collection_name="test",
        points=[
            PointStruct(id=1, vector=np.random.rand(384).tolist(), payload={"text": "Hello"}),
            PointStruct(id=2, vector=np.random.rand(384).tolist(), payload={"text": "World"}),
        ]
    )

    print("Qdrant test passed")
except ImportError:
    print("Qdrant client not installed")

## 4. Advanced: Metadata Filtering

In [ ]:
class VectorDBWithFilter:
    def __init__(self):
        self.documents = []
        self.embeddings = None
        self.metadata = []
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def add(self, documents: list[str], metadata: list[dict] | None = None):
        """Add documents with metadata."""

        new_embeddings = self.embedder.encode(documents)
        self.documents.extend(documents)
        self.metadata.extend(metadata or [{}] * len(documents))

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search_with_filter(self, query: str, filter_key: str = None, filter_value: str = None, top_k: int = 5):
        """Search with optional metadata filter."""

        if self.embeddings is None:
            return []

        # TODO: Apply filter if specified
        valid_indices = list(range(len(self.documents)))

        if filter_key and filter_value:
            valid_indices = [
                i for i in range(len(self.metadata))
                if self.metadata[i].get(filter_key) == filter_value
            ]

        # Search within valid indices
        query_emb = self.embedder.encode([query])[0]
        valid_embeddings = self.embeddings[valid_indices]

        similarities = np.dot(valid_embeddings, query_emb)
        top_local_indices = similarities.argsort()[-top_k:][::-1]

        results = []
        for local_idx in top_local_indices:
            global_idx = valid_indices[local_idx]
            results.append((self.documents[global_idx], similarities[local_idx]))

        return results

# Test
db = VectorDBWithFilter()
db.add(
    ["Python tutorial", "Java guide", "Python tips"],
    [{"lang": "python"}, {"lang": "java"}, {"lang": "python"}]
)

results = db.search_with_filter("programming", filter_key="lang", filter_value="python")
for doc, score in results:
    print(f"[{score:.3f}] {doc}")

## Exercise

1. Implement hybrid search (dense + sparse)
2. Add document re-ranking
3. Implement batch operations